<a href="https://colab.research.google.com/github/veigaeduarda/PAnaM_Webscraping_de_Jornais/blob/main/O_JOIO_E_O_TRIGO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import ssl # Importação necessária para o contexto não-verificado
from urllib.request import urlopen, Request
from urllib.error import HTTPError
from urllib.parse import quote_plus

In [ ]:
eixo_1 = ['política','programa','lei','regulação','restrição','taxação','imposto','tributo','tributação']
eixo_2 = ['política','programa','subsídio',"'incentivo fiscal'","'incentivos fiscais'","'isenção fiscal'"]
eixo_3 = ['política','programa','lei','regulação']
eixo_4 = ['política','programa','lei','regulação','restrição']
eixo_5 = ['política','programa','lei','regulação','restrição']
eixo_6 = ['política','programa','lei','regulação','restrição','imposto','taxação']

assunto_1 = ['ultraprocessado','ultraprocessados',"'alimentos industrializados'",'gordura','açúcar','aditivo','edulcorante',
              'adoçante','flavorizante','conservante','ultraprocessada','ultraprocessadas',"'bebidas açucaradas'",'refrigerante',
              'açúcares','aditivos','edulcorantes','adoçantes','flavorizantes','refrigerantes','bebidas ultraprocessadas']
assunto_2 = ["'alimentos saudáveis'",'frutas','verduras',"'cesta básica'"]
assunto_3 = ["'rotulagem de alimentos'","'rótulo alimentar'","'rótulo nutricional'","'rótulo frontal'"]
assunto_4 = ["'publicidade de alimentos'","'publicidade de bebidas'","'marketing de alimentos'","'marketing de bebidas'",
              "'propaganda de alimentos'","'propaganda de bebidas'"]
assunto_5 = ["'alimentação escolar'",'merenda',"'cantina escolar'","'lanche escolar'","'refeição escolar'"]
assunto_6 = ['pesticidas','agrotóxicos','inseticidas','agroquímicos',"'defensivos agrícolas'"]

policies_br = ["Programa Nacional de Alimentação Escolar", "Programa de Aquisição de Alimentos"]

In [ ]:
import unicodedata

def remove_accents_and_cedilla(text):
    # Normalize to NFD (canonical decomposition) and encode to ASCII, ignoring errors (removes accents)
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    # Replace 'ç' with 'c' and 'Ç' with 'C'
    text = text.replace('ç', 'c').replace('Ç', 'C')
    return text

queries = []

for i in range(1, 7):
    eixo_list = globals()[f'eixo_{i}']
    assunto_list = globals()[f'assunto_{i}']
    for eixo_term in eixo_list:
        for assunto_term in assunto_list:
            queries.append(f'{eixo_term} {assunto_term}')

# Adicionar as políticas brasileiras como queries
queries.extend(policies_br)

# Aplicar remoção de acentos e 'ç'
queries = [remove_accents_and_cedilla(query) for query in queries]

# Remover aspas simples extras e substituir espaços por '+' nas queries
queries = [query.replace("'", "").replace(" ", "+") for query in queries]

print(f"Total de queries geradas: {len(queries)}")
print(queries)

Total de queries geradas: 321
['politica+ultraprocessado', 'politica+ultraprocessados', 'politica+alimentos+industrializados', 'politica+gordura', 'politica+acucar', 'politica+aditivo', 'politica+edulcorante', 'politica+adocante', 'politica+flavorizante', 'politica+conservante', 'politica+ultraprocessada', 'politica+ultraprocessadas', 'politica+bebidas+acucaradas', 'politica+refrigerante', 'politica+acucares', 'politica+aditivos', 'politica+edulcorantes', 'politica+adocantes', 'politica+flavorizantes', 'politica+refrigerantes', 'politica+bebidas+ultraprocessadas', 'programa+ultraprocessado', 'programa+ultraprocessados', 'programa+alimentos+industrializados', 'programa+gordura', 'programa+acucar', 'programa+aditivo', 'programa+edulcorante', 'programa+adocante', 'programa+flavorizante', 'programa+conservante', 'programa+ultraprocessada', 'programa+ultraprocessadas', 'programa+bebidas+acucaradas', 'programa+refrigerante', 'programa+acucares', 'programa+aditivos', 'programa+edulcorantes', 

In [ ]:
urls = []
busca = []
site = []

for query in queries:

    for n in range(1, 10):

        # página da busca
        # Encode the query string to handle spaces and other special characters
        encoded_query = quote_plus(query)
        url = f"https://ojoioeotrigo.com.br/page/{n}/?s={encoded_query}"

        print(url)

        try:
            req = Request(
                url,
                headers={'User-Agent': 'Mozilla/5.0'}
            )

            html = urlopen(req)

            bsObj = BeautifulSoup(html.read(), 'html.parser')

            links_encontrados = 0

            for link in bsObj.find_all('a'):

                url2 = link.get('href')

                if (
                    url2 is not None
                    and "ojoioeotrigo.com.br/" in url2
                    and "/202" in url2
                ):

                    links_encontrados += 1

                    if url2 not in urls:

                        print(url2)

                        urls.append(url2)
                        site.append("Joio")
                        busca.append(query)

            # se não encontrou nenhuma matéria, para
            if links_encontrados == 0:

                print(f"Sem resultados na página {n}")
                break

        except HTTPError as e:

            if e.code == 404:

                print(f"Página {n} não existe")
                break

            else:
                print(e)

        except Exception as e:
            print(e)

joio = pd.DataFrame({
    'Urls': urls,
    'Sites': site,
    'Buscas': busca
})

print(joio.head())

https://ojoioeotrigo.com.br/page/1/?s=politica%2Bultraprocessado
https://ojoioeotrigo.com.br/2025/03/faz-sentido-falar-em-ultraprocessados-menos-piores/
https://ojoioeotrigo.com.br/2024/10/escolas-livres-de-ultraprocessados-sao-uma-realidade-inevitavel/
https://ojoioeotrigo.com.br/2025/10/quando-a-ciencia-opera-em-favor-dos-ultraprocessados/
https://ojoioeotrigo.com.br/2022/03/ultraprocessado-nosso-de-cada-dia-a-doenca-chega-embalada-na-cidade/
https://ojoioeotrigo.com.br/2024/10/as-corporacoes-tem-como-alimento-o-dinheiro/
https://ojoioeotrigo.com.br/2020/12/bh-vive-explosao-de-ultraprocessados-em-dez-anos-estabelecimentos-nao-saudaveis-cresceram-154-e-delivery-7-000/
https://ojoioeotrigo.com.br/2024/11/margarina-isenta-de-impostos-e-pronta-para-sentar-a-mesa/
https://ojoioeotrigo.com.br/2025/07/como-a-industria-molda-projetos-de-lei-para-manter-ultraprocessados-nas-escolas/
https://ojoioeotrigo.com.br/page/2/?s=politica%2Bultraprocessado
https://ojoioeotrigo.com.br/2022/08/politica-d

In [ ]:
print (joio)

                                                  Urls Sites  \
0    https://ojoioeotrigo.com.br/2025/03/faz-sentid...  Joio   
1    https://ojoioeotrigo.com.br/2024/10/escolas-li...  Joio   
2    https://ojoioeotrigo.com.br/2025/10/quando-a-c...  Joio   
3    https://ojoioeotrigo.com.br/2022/03/ultraproce...  Joio   
4    https://ojoioeotrigo.com.br/2024/10/as-corpora...  Joio   
..                                                 ...   ...   
544  https://ojoioeotrigo.com.br/2021/11/agronegoci...  Joio   
545  https://ojoioeotrigo.com.br/2020/03/coronaviru...  Joio   
546  https://ojoioeotrigo.com.br/2022/04/titula-bra...  Joio   
547  https://ojoioeotrigo.com.br/2021/08/dieta-ente...  Joio   
548  https://ojoioeotrigo.com.br/2022/06/33-milhoes...  Joio   

                                 Buscas  
0              politica+ultraprocessado  
1              politica+ultraprocessado  
2              politica+ultraprocessado  
3              politica+ultraprocessado  
4              politi

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import ssl
import re
from urllib.request import urlopen, Request

# Reiniciando o DataFrame para garantir limpeza
dados = pd.DataFrame(columns=['site', 'data', 'titulo', 'editoria', 'url', 'texto', 'autor'])

ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

for index, row in joio.iterrows():
    link = row['Urls']
    original_query = row['Buscas']

    data_row_dict = {
        'site': 'Joio e o Trigo',
        'data': 'Não obtido',
        'titulo': 'Não obtido',
        'editoria': original_query,
        'url': link,
        'texto': 'Não obtido',
        'autor': 'Não obtido'
    }

    try:
        req = Request(link, headers={'User-Agent': 'Mozilla/5.0'})
        html = urlopen(req, context=ssl_context)
        bsObj = BeautifulSoup(html.read(), 'html.parser')

        # Titulo
        title_tag = bsObj.find(['h1', 'h2'], class_=['entry-title', 'post-title'])
        if title_tag:
            data_row_dict['titulo'] = title_tag.get_text(strip=True)

        # Autor
        author_tag = bsObj.find('span', class_=['author', 'vcard', 'fn', 'author-name'])
        if not author_tag:
            author_tag = bsObj.find('a', rel='author')
        if not author_tag:
            author_tag = bsObj.select_one('.author-info .author-name')

        if author_tag:
            author_text = author_tag.get_text(strip=True).replace('Por ', '').replace('por ', '')
            data_row_dict['autor'] = author_text

        # Data - Tenta extrair do HTML
        date_tag = bsObj.find('time', class_=['entry-date', 'published', 'updated'])
        if date_tag:
            if date_tag.has_attr('datetime'):
                data_row_dict['data'] = date_tag['datetime'].split('T')[0]
            else:
                data_row_dict['data'] = date_tag.get_text(strip=True)

        # Fallback: Extrair data da URL se ainda estiver 'Não obtido'
        if data_row_dict['data'] == 'Não obtido':
            match = re.search(r'/(\d{4})/(\d{2})/', link)
            if match:
                data_row_dict['data'] = f"{match.group(1)}-{match.group(2)}"

        # Texto
        content_div = bsObj.find('div', class_=['entry-content', 'post-content', 'article-content'])
        if content_div:
            paragraphs = content_div.find_all('p')
            data_row_dict['texto'] = '\n'.join([p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)])

    except Exception as e:
        print(f"Erro no link {link}: {e}")

    print(f"Processado: {link} | Data: {data_row_dict['data']} | Autor: {data_row_dict['autor']}")
    dados = pd.concat([dados, pd.DataFrame([data_row_dict])], ignore_index=True)

display(dados.head())

Processado: https://ojoioeotrigo.com.br/2025/03/faz-sentido-falar-em-ultraprocessados-menos-piores/ | Data: 2025-03 | Autor: João Peres
Processado: https://ojoioeotrigo.com.br/2024/10/escolas-livres-de-ultraprocessados-sao-uma-realidade-inevitavel/ | Data: 2024-10 | Autor: Luiza Sansão
Processado: https://ojoioeotrigo.com.br/2025/10/quando-a-ciencia-opera-em-favor-dos-ultraprocessados/ | Data: 2025-10 | Autor: Rodrigo Oliveira
Processado: https://ojoioeotrigo.com.br/2022/03/ultraprocessado-nosso-de-cada-dia-a-doenca-chega-embalada-na-cidade/ | Data: 2022-03 | Autor: Anelize Moreira
Processado: https://ojoioeotrigo.com.br/2024/10/as-corporacoes-tem-como-alimento-o-dinheiro/ | Data: 2024-10 | Autor: Maíra Mathias
Processado: https://ojoioeotrigo.com.br/2020/12/bh-vive-explosao-de-ultraprocessados-em-dez-anos-estabelecimentos-nao-saudaveis-cresceram-154-e-delivery-7-000/ | Data: 2020-12 | Autor: Mylena Melo
Processado: https://ojoioeotrigo.com.br/2024/11/margarina-isenta-de-impostos-e-pro

,site,data,titulo,editoria,url,texto,autor
0,Joio e o Trigo,2025-03,Faz sentido falar em ultraprocessados menos pi...,politica+ultraprocessado,https://ojoioeotrigo.com.br/2025/03/faz-sentid...,Poderia ser um debate saudável em vários senti...,João Peres
1,Joio e o Trigo,2024-10,Escolas livres de ultraprocessados são uma rea...,politica+ultraprocessado,https://ojoioeotrigo.com.br/2024/10/escolas-li...,“A gente nunca mais viu criança nenhuma chegar...,Luiza Sansão
2,Joio e o Trigo,2025-10,Quando a ciência opera em favor dos ultraproce...,politica+ultraprocessado,https://ojoioeotrigo.com.br/2025/10/quando-a-c...,Se hoje um ultraprocessado é chamado pelo nome...,Rodrigo Oliveira
3,Joio e o Trigo,2022-03,Ultraprocessado nosso de cada dia: a doença ch...,politica+ultraprocessado,https://ojoioeotrigo.com.br/2022/03/ultraproce...,Os pseudoalimentos vendidos em pacotes moderno...,Anelize Moreira
4,Joio e o Trigo,2024-10,“As corporações têm como alimento o dinheiro”,politica+ultraprocessado,https://ojoioeotrigo.com.br/2024/10/as-corpora...,Chris van Tulleken quer construir pontes. O au...,Maíra Mathias


In [ ]:
from google.colab import files

# Definindo as colunas desejadas
colunas = ['site', 'data', 'titulo', 'editoria', 'url', 'texto', 'autor']

# Criando o CSV a partir do DataFrame 'dados'
# Usamos index=False para não incluir a coluna de índices no arquivo
dados[colunas].to_csv('dados_coletados.csv', index=False, encoding='utf-8-sig')

# Iniciando o download do arquivo no navegador
files.download('dados_coletados.csv')

print("Arquivo 'dados_coletados.csv' gerado e pronto para download!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Arquivo 'dados_coletados.csv' gerado e pronto para download!
